# Tunnel: is sensor 1 resolvable?

The tunnel's unweighted metric averages R² over eleven sensors with equal weight. Sensor 1
sits at the tube end where the response is two orders of magnitude smaller than mid-span, so
its R² is dominated by a near-zero denominator rather than by prediction quality. This
notebook quantifies that and reports the tunnel row over the ten remaining sensors.

**No training and no GPU.** Arm 2 and Arm 3 per-sensor R² are already stored in each result
JSON; Arm 1 needs no model and is recomputed from the database on the same split.

Drop this notebook into the same Drive folder as the sensitivity sweep and Run all.


## 1 · Point at the existing sweep

In [ ]:
import os

from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/colab_sensitivity"
OUTD = os.path.join(BASE, "results")
DATA = os.path.join(BASE, "data", "tunnel")

assert os.path.isdir(OUTD), f"no results at {OUTD}"
assert os.path.isdir(DATA), f"no tunnel databases at {DATA}"
LEVELS = {1: "Q 3.0", 2: "Q 1.8", 3: "Q 1.0", 4: "Q 0.6", 5: "Q 0.35"}
SEEDS = [42, 0, 1, 2, 3]
print("results:", len([f for f in os.listdir(OUTD) if f.startswith("tunnel_")]), "tunnel files")

## 2 · Evidence: what sensor 1 actually measures

Arm 1 is the reduced model against the reference, so it needs no training. The split is
`np.random.default_rng(seed).permutation(N)` cut 70/15/15 — the same one the sweep used, and
reproducible from the seed alone, so this cell does not depend on the checkpoints.


In [ ]:
import numpy as np

def split_te(n, seed):
    idx = np.random.default_rng(seed).permutation(n)
    return idx[int(0.70 * n) + int(0.15 * n):]

def r2(y, p):
    return float(1 - ((y - p) ** 2).sum() / (((y - y.mean()) ** 2).sum() + 1e-12))

ARM1 = {}      # (level, seed) -> list of 11 per-sensor R2
AMP = {}       # level -> list of 11 RMS amplitudes [mm]
for lv in LEVELS:
    d = np.load(os.path.join(DATA, f"L{lv}.npz"), allow_pickle=True)
    nl = np.transpose(d["u_nl"], (0, 2, 1)); rd = np.transpose(d["u_red"], (0, 2, 1))
    AMP[lv] = [float(np.sqrt((nl[..., s] ** 2).mean())) * 1e3 for s in range(nl.shape[2])]
    for sd in SEEDS:
        te = split_te(nl.shape[0], sd)
        ARM1[(lv, sd)] = [r2(nl[te][..., s], rd[te][..., s]) for s in range(nl.shape[2])]

print("RMS amplitude per sensor [mm], and Arm 1 R2 (seed 42)\n")
print(f"{'level':>7} " + "".join(f"{'s'+str(s+1):>9}" for s in range(11)))
for lv, kn in LEVELS.items():
    print(f"{kn:>7} " + "".join(f"{AMP[lv][s]:9.3f}" for s in range(11)))
print()
for lv, kn in LEVELS.items():
    print(f"{kn:>7} " + "".join(f"{ARM1[(lv,42)][s]:9.2f}" for s in range(11)))
print("\nsensor 1 amplitude relative to the loudest sensor:")
for lv, kn in LEVELS.items():
    print(f"  {kn:>7}  {100*AMP[lv][0]/max(AMP[lv]):5.2f}%")

## 3 · Re-score: all eleven sensors vs sensors 2–11

Arm 2 and Arm 3 come from the stored `r2_per_sensor`. Nothing is retrained, so the two
columns differ only in which sensors are averaged.


In [ ]:
import glob, json
import statistics as st

R = {}
for f in glob.glob(os.path.join(OUTD, "tunnel_L*_arm*_s*.json")):
    r = json.load(open(f))
    R[(r["level"], r["arm"], r["seed"])] = r["r2_per_sensor"]

def unw(v, drop_first):
    v = v[1:] if drop_first else v
    return sum(v) / len(v)

def agg(vals):
    return (st.mean(vals), st.stdev(vals) if len(vals) > 1 else 0.0)

rows = []
for drop in (False, True):
    lab = "sensors 2-11" if drop else "all 11 sensors"
    print(f"\n=== {lab} ===")
    print(f"{'level':>7} {'Arm 1':>17} {'Arm 2':>17} {'Arm 3':>17} {'A3>A2':>6}")
    for lv, kn in LEVELS.items():
        have = [s for s in SEEDS if (lv, 3, s) in R and (lv, 2, s) in R]
        if len(have) < len(SEEDS):
            print(f"{kn:>7}  incomplete: {len(have)}/{len(SEEDS)} seeds"); continue
        a1 = [unw(ARM1[(lv, s)], drop) for s in have]
        a2 = [unw(R[(lv, 2, s)], drop) for s in have]
        a3 = [unw(R[(lv, 3, s)], drop) for s in have]
        wins = sum(x > y for x, y in zip(a3, a2))
        f = lambda v: f"{agg(v)[0]:+.4f} +/- {agg(v)[1]:.4f}"
        print(f"{kn:>7} {f(a1):>17} {f(a2):>17} {f(a3):>17} {wins}/{len(have)}")
        rows.append([lab, lv, kn, f"{agg(a1)[0]:.4f}", f"{agg(a1)[1]:.4f}",
                     f"{agg(a2)[0]:.4f}", f"{agg(a2)[1]:.4f}",
                     f"{agg(a3)[0]:.4f}", f"{agg(a3)[1]:.4f}", f"{wins}/{len(have)}"])

## 4 · What dropping sensor 1 changes

In [ ]:
import csv

print(f"{'level':>7} {'Arm 1 all':>10} {'Arm 1 2-11':>11} {'shift':>8} | "
      f"{'Arm 3 all':>10} {'Arm 3 2-11':>11} {'shift':>8}")
for lv, kn in LEVELS.items():
    have = [s for s in SEEDS if (lv, 3, s) in R]
    if len(have) < len(SEEDS):
        continue
    a1a = st.mean([unw(ARM1[(lv, s)], False) for s in have])
    a1b = st.mean([unw(ARM1[(lv, s)], True) for s in have])
    a3a = st.mean([unw(R[(lv, 3, s)], False) for s in have])
    a3b = st.mean([unw(R[(lv, 3, s)], True) for s in have])
    print(f"{kn:>7} {a1a:10.4f} {a1b:11.4f} {a1b-a1a:+8.4f} | "
          f"{a3a:10.4f} {a3b:11.4f} {a3b-a3a:+8.4f}")

p = os.path.join(OUTD, "tunnel_sensor1_rescored.csv")
with open(p, "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["metric", "level", "knob", "arm1_mean", "arm1_sd",
                "arm2_mean", "arm2_sd", "arm3_mean", "arm3_sd", "arm3_beats_arm2"])
    w.writerows(rows)
print("\nwrote", p)

---

### How to read the result

If the tunnel's apparent improvement from level 1 to level 5 largely disappears once sensor 1
is removed, then that trend was a property of the metric rather than of the method: sensor 1
moves about 10 µm at Q 3.0 and about 70 µm at Q 0.35, so its R² recovers as the denominator
grows, and it carries one vote in eleven.

If the trend survives, the improvement is real and distributed across the resolvable sensors.

### Caveat

Dropping a sensor departs from the definition used in Table 2, so the two cannot be compared
directly. Report it as a robustness check on the tunnel row, not as a replacement for it, and
state the amplitude figures that justify the exclusion — an R² of −8.35 on a sensor moving
10 µm is a statement about resolution, not about accuracy.
